# Complete Experiment Pipeline

Run the complete experiment: probe training → visualization → interventions → interpretation


In [ ]:
import sys
import os
from pathlib import Path

# Ensure we're in the right directory
if not os.path.exists('src'):
    if os.path.exists('Negation-Origin-Tracing'):
        os.chdir('Negation-Origin-Tracing')
    else:
        print("⚠ Warning: Could not find project directory")
        print(f"  Current: {os.getcwd()}")
        print(f"  Run 00_setup_colab.ipynb first!")

sys.path.insert(0, os.getcwd())
print(f"✓ Working directory: {os.getcwd()}")


In [ ]:
# Configuration
import torch

# Auto-detect GPU
use_gpu = torch.cuda.is_available()

config = {
    'data_dir': 'data/raw',
    'model_name': 'distilbert-base-uncased',
    'batch_size': 32 if use_gpu else 16,
    'max_epochs': 10,
    'probe_lr': 1e-3,
    'seed': 42,
    'layers': 'all',
    'pooling_strategies': 'all',
    'output_dir': 'experiments/full_experiment',
    'top_k': 5,
    'skip_search': False,  # Set to True to use existing probe results
    'skip_interventions': False,  # Set to True to skip interventions
    'devices': 1 if use_gpu else 1,
}

print("Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

if use_gpu:
    print(f"\n✓ Will use GPU: {torch.cuda.get_device_name(0)}")
else:
    print("\n⚠ No GPU - experiment will take longer. Consider enabling GPU in Colab.")
    
print("\nNote: This will run all experiments. It may take 1-2 hours depending on GPU.")


In [ ]:
# Run complete experiment pipeline
import subprocess

cmd = [
    'python', 'src/scripts/run_full_experiment.py',
    '--data_dir', config['data_dir'],
    '--model_name', config['model_name'],
    '--batch_size', str(config['batch_size']),
    '--max_epochs', str(config['max_epochs']),
    '--probe_lr', str(config['probe_lr']),
    '--output_dir', config['output_dir'],
    '--layers', config['layers'],
    '--pooling_strategies', config['pooling_strategies'],
    '--seed', str(config['seed']),
    '--top_k', str(config['top_k']),
    '--devices', str(config['devices']),
]

if config['skip_search']:
    cmd.append('--skip_search')
if config['skip_interventions']:
    cmd.append('--skip_interventions')

print("Running complete experiment pipeline...")
print(f"Command: {' '.join(cmd)}")
result = subprocess.run(cmd, check=True)
print("\n✓ Experiment complete!")


In [ ]:
# Display interpretation report
import json

report_path = os.path.join(config['output_dir'], 'interpretation_report.json')

if os.path.exists(report_path):
    with open(report_path, 'r') as f:
        report = json.load(f)
    
    print("\n" + "="*60)
    print("INTERPRETATION REPORT")
    print("="*60)
    
    print(f"\nBest Layer: {report['summary']['best_layer']}")
    print(f"Best Probe AUROC: {report['summary']['best_probe_auroc']:.4f}")
    
    print(f"\nTop {config['top_k']} Negation Layers:")
    for i, layer in enumerate(report['negation_layers'], 1):
        print(f"  {i}. Layer {layer['layer_idx']}: "
              f"score={layer['composite_score']:.4f}, "
              f"AUROC={layer['avg_probe_score']:.4f}, "
              f"best_pooling={layer['best_pooling_strategy']}")
    
    if report['causality_verification']:
        causal_count = sum(1 for c in report['causality_verification'] if c['is_causal'])
        print(f"\nCausal Layers Found: {causal_count}/{len(report['causality_verification'])}")
    
    print("\nRecommendations:")
    for rec in report['recommendations']:
        print(f"  - {rec}")
else:
    print(f"⚠ Report not found at {report_path}")
